<a href="https://colab.research.google.com/github/zogydinesh/ITA-0616/blob/main/candidate_elimination.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
data = """Size,Color,Shape,Class
Big,Red,Circle,No
Small,Red,Triangle,No
Small,Red,Circle,Yes
Big,Blue,Circle,No
Small,Blue,Circle,Yes"""

with open('training_data.csv', 'w') as f:
    f.write(data)

In [5]:
import csv

def covers(h, x):
    """Does hypothesis h match example x?"""
    return all(a == '?' or a == b for a, b in zip(h, x))

def more_general(h1, h2):
    """Is h1 at least as general as h2?"""
    return all(a == '?' or b == '0' or a == b for a, b in zip(h1, h2))

# ---- 1. Load CSV ----
with open('training_data.csv') as f:
    rows = list(csv.reader(f))
    header, data = rows[0], rows[1:]
    n = len(header) - 1                      # number of attributes
    domains = [sorted(set(r[i] for r in data)) for i in range(n)]

    # ---- 2. Initialise S and G ----
    S = ['0'] * n
    G = [['?'] * n]
    print("Initial S:", S)
    print("Initial G:", G, "\n")

    # ---- 3. Process each example ----
    for step, row in enumerate(data, 1):
        inst, label = row[:-1], row[-1].strip().lower()

        if label == 'yes':                    # positive example
            G = [g for g in G if covers(g, inst)]
            for i in range(n):
                if S[i] == '0':
                    S[i] = inst[i]
                elif S[i] != inst[i]:
                    S[i] = '?'
        else:                                 # negative example
            new_G = []
            for g in G:
                if not covers(g, inst):
                    new_G.append(g)
                else:
                    for i in range(n):
                        if g[i] == '?':
                            for v in domains[i]:
                                if v != inst[i]:
                                    h = g.copy()
                                    h[i] = v
                                    if more_general(h, S):
                                        new_G.append(h)
            # remove duplicates and non-maximal hypotheses
            uniq = [list(t) for t in set(tuple(g) for g in new_G)]
            G = [g for g in uniq if not any(o != g and more_general(o, g) for o in uniq)]

        print(f"Example {step}: {inst} -> {row[-1]}")
        print("  S =", S)
        print("  G =", G, "\n")

    # ---- 4. Final result ----
    print("Final Specific Hypothesis S:", S)
    print("Final General Hypotheses  G:", G)

Initial S: ['0', '0', '0']
Initial G: [['?', '?', '?']] 

Example 1: ['Big', 'Red', 'Circle'] -> No
  S = ['0', '0', '0']
  G = [['?', '?', 'Triangle'], ['?', 'Blue', '?'], ['Small', '?', '?']] 

Example 2: ['Small', 'Red', 'Triangle'] -> No
  S = ['0', '0', '0']
  G = [['Big', '?', 'Triangle'], ['?', 'Blue', '?'], ['Small', '?', 'Circle']] 

Example 3: ['Small', 'Red', 'Circle'] -> Yes
  S = ['Small', 'Red', 'Circle']
  G = [['Small', '?', 'Circle']] 

Example 4: ['Big', 'Blue', 'Circle'] -> No
  S = ['Small', 'Red', 'Circle']
  G = [['Small', '?', 'Circle']] 

Example 5: ['Small', 'Blue', 'Circle'] -> Yes
  S = ['Small', '?', 'Circle']
  G = [['Small', '?', 'Circle']] 

Final Specific Hypothesis S: ['Small', '?', 'Circle']
Final General Hypotheses  G: [['Small', '?', 'Circle']]
